Making Network1 html:

In [39]:
import pandas as pd

df = pd.read_csv("about_networks/genre_genre.csv")
print(df.columns)

Index(['Source', 'Target', 'Weight'], dtype='str')


In [43]:
from pyvis.network import Network
import pandas as pd
import networkx as nx

# ----------------------------
# 1. Load CSV
# ----------------------------
df = pd.read_csv("about_networks/genre_genre.csv")
df.columns = df.columns.str.strip()

# ----------------------------
# 2. Create NetworkX Graph
# ----------------------------
G = nx.from_pandas_edgelist(df, source="Source", target="Target", edge_attr="Weight")

# ----------------------------
# 3. Create PyVis Network
# ----------------------------
net = Network(height="800px", width="100%", bgcolor="#111111", font_color="white", notebook=False)
net.from_nx(G)

# ----------------------------
# 4. Physics + Hover Effects
# ----------------------------
net.set_options("""
{
  "nodes": {
    "borderWidth": 2,
    "size": 15,
    "color": {
      "border": "#ffffff",
      "background": "#1f78b4",
      "highlight": {
        "border": "#ffffff",
        "background": "#4db8ff"
      },
      "hover": {
        "border": "#ffffff",
        "background": "#4db8ff"
      }
    },
    "font": {
      "size": 14,
      "face": "Tahoma",
      "color": "#ffffff"
    },
    "physics": true
  },
  "edges": {
    "color": {
      "color": "#1f78b4",
      "highlight": "#03fc35"
    },
    "smooth": {
      "type": "continuous"
    }
  },
  "physics": {
    "enabled": true,
    "barnesHut": {
      "gravitationalConstant": -2500,
      "centralGravity": 0.15,
      "springLength": 250,
      "springConstant": 0.005,
      "damping": 0.35,
      "avoidOverlap": 0.5
    },
    "minVelocity": 0.02,
    "solver": "barnesHut",
    "stabilization": {
      "enabled": false
    }
  },
  "interaction": {
    "hover": true,
    "tooltipDelay": 100,
    "hoverConnectedEdges": true,
    "multiselect": true,
    "selectable": true
  }
}
""")

# ----------------------------
# 5. Export Interactive HTML
# ----------------------------
html_file = "genre_genre_network.html"
net.write_html(html_file, notebook=False)

# ----------------------------
# 6. Inject custom JS for click highlight + title + info panel
# ----------------------------
custom_js = """
<script type="text/javascript">
(function() {
    var nodes = network.body.data.nodes;
    var edges = network.body.data.edges;

    // --- Click highlight ---
    network.on("selectNode", function(params) {
        for (var i = 0; i < params.nodes.length; i++) {
            var nodeId = params.nodes[i];
            var node = nodes.get(nodeId);
            node.color.background = "#91cefc"; // dark green on click
            nodes.update(node);
        }

        // --- Update info panel ---
        var nodeId = params.nodes[0];
        if (!nodeId) return;
        var connectedEdges = network.getConnectedEdges(nodeId);
        infoDiv.innerHTML = "<b>Node:</b> " + nodeId + " | <b>Edges:</b> " + connectedEdges.length;
    });

    network.on("deselectNode", function(params) {
        nodes.get().forEach(function(node) {
            if (!node.selected) {
                node.color.background = "#1f78b4"; // reset to original
                nodes.update(node);
            }
        });
        infoDiv.innerHTML = "";
    });

    // --- Info panel below canvas ---
    var infoDiv = document.createElement('div');
    infoDiv.style.position = 'absolute';
    infoDiv.style.bottom = '20px';
    infoDiv.style.left = '50%';
    infoDiv.style.transform = 'translateX(-50%)';
    infoDiv.style.color = '#ffffff';
    infoDiv.style.fontSize = '16px';
    infoDiv.style.fontFamily = 'Tahoma';
    infoDiv.style.backgroundColor = 'rgba(0,0,0,0.6)';
    infoDiv.style.padding = '8px 12px';
    infoDiv.style.borderRadius = '6px';
    infoDiv.style.pointerEvents = 'none';
    infoDiv.style.transition = 'all 0.3s ease-in-out';
    infoDiv.style.opacity = 0;
    document.body.appendChild(infoDiv);

    network.on("selectNode", function() { infoDiv.style.opacity = 1; });
    network.on("deselectNode", function() { infoDiv.style.opacity = 0; });

    // --- Network title above canvas ---
    var titleDiv = document.createElement('div');
    titleDiv.innerHTML = "Genre-Genre Undirected Co-Occurance Network";
    titleDiv.style.position = 'absolute';
    titleDiv.style.top = '20px';  // top margin
    titleDiv.style.left = '50%';
    titleDiv.style.transform = 'translateX(-50%)';
    titleDiv.style.color = '#4db8ff';
    titleDiv.style.fontSize = '18px';
    titleDiv.style.fontWeight = 'bold';
    titleDiv.style.fontFamily = 'Tahoma';
    titleDiv.style.backgroundColor = 'transparent'; // no background, fully black behind
    titleDiv.style.border = 'none';  // remove any borders
    titleDiv.style.pointerEvents = 'none';
    document.body.appendChild(titleDiv);
})();
</script>
"""

# Append JS to HTML
with open(html_file, "r", encoding="utf-8") as f:
    html_content = f.read()

html_content = html_content.replace("</body>", custom_js + "\n</body>")

with open(html_file, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"Interactive network saved to {html_file} with clean black background and node edges info!")

Interactive network saved to genre_genre_network.html with clean black background and node edges info!


Making Network2 html:

In [41]:
import pandas as pd

df = pd.read_csv("about_networks/anime_community_assignments.csv")
print(df.columns)
df["is_bridge"].value_counts()

Index(['anime_id', 'anime_name', 'genres', 'num_genres',
       'assigned_communities', 'all_community_names', 'confidence',
       'is_bridge', 'communities_spanned', 'community_distribution'],
      dtype='str')


is_bridge
False    13060
True      3091
Name: count, dtype: int64

In [59]:
from pyvis.network import Network
import pandas as pd
import networkx as nx

# ----------------------------
# 1. Load Data
# ----------------------------
df = pd.read_csv("about_networks/anime_community_assignments.csv")
df.columns = df.columns.str.strip()

# ----------------------------
# 2. Sample Data for speed
# ----------------------------
bridge_df = df[df["is_bridge"] == True]
non_bridge_df = df[df["is_bridge"] == False]

# Take all bridges + 10% of non-bridges
non_bridge_sample = non_bridge_df.sample(frac=0.1, random_state=42)
final_df = pd.concat([bridge_df, non_bridge_sample])

# ----------------------------
# 3. Count anime per community for sizing
# ----------------------------
community_colors = {
    "Action & Adventure": "#9145b0",
    "Romance & Slice of Life": "#0fceaa",
    "Dark & Psychological": "#f37cd6",
    "Niche & Fantasy": "#0e149e"
}

community_counts = {comm: 0 for comm in community_colors}
for _, row in final_df.iterrows():
    communities = [c.strip() for c in str(row["all_community_names"]).split(",")]
    for comm in communities:
        if comm in community_counts:
            community_counts[comm] += 1

# ----------------------------
# 4. Gradient for bridge anime
# ----------------------------
def bridge_color(num_communities):
    """Map number of communities to color from yellow → orange → red"""
    if num_communities == 1:
        return "#4db8ff"  # blue 1
    elif num_communities == 2:
        return "#61df90"  # green 2
    elif num_communities == 3:
        return "#ffb347"  # orange 3
    else:
        return "#ff3333"  # red for 4+

# ----------------------------
# 5. Build Graph
# ----------------------------
G = nx.Graph()

# Add community nodes
for comm, color in community_colors.items():
    size = 70 + community_counts[comm] // 5
    G.add_node(
        comm,
        node_type="community",
        color=color,
        size=size,
        physics=False,
        label=comm,
        title=f"<b>Community:</b> {comm}<br><b>#Anime:</b> {community_counts[comm]}"
    )

# Add anime nodes + edges
for _, row in final_df.iterrows():
    anime = row["anime_name"]
    communities = [c.strip() for c in str(row["all_community_names"]).split(",")]
    num_spanned = len(communities)
    size = 10 if num_spanned == 1 else 10 + 5 * (num_spanned - 1)

    # Color based on bridge status
    color = bridge_color(num_spanned)
    
    # Show actual communities in tooltip
    tooltip = f"<b>Anime:</b> {anime}<br>" \
              f"<b>Communities Spanned ({num_spanned}):</b> {', '.join(communities)}"

    G.add_node(
        anime,
        node_type="anime",
        base_color=color,
        color=color,
        size=size,
        physics=True,
        title=tooltip
    )
    
    for comm in communities:
        if comm in community_colors:
            G.add_edge(anime, comm)

# ----------------------------
# 6. Create PyVis Network
# ----------------------------
net = Network(
    height="800px",
    width="100%",
    bgcolor="#111111",
    font_color="white",
    notebook=False
)
net.from_nx(G)

# ----------------------------
# 7. Fix Community Positions
# ----------------------------
positions = {
    "Action & Adventure": (-8000, 5000),
    "Romance & Slice of Life": (8000, 5000),
    "Dark & Psychological": (-8000, -5000),
    "Niche & Fantasy": (8000, 5000)  # closer to center
}

for node in net.nodes:
    if node["id"] in positions:
        node["x"], node["y"] = positions[node["id"]]
        node["physics"] = False

# ----------------------------
# 8. Initial Options
# ----------------------------
net.set_options("""
{
  "nodes": {
    "borderWidth": 2,
    "font": {
      "size": 12,
      "face": "Tahoma",
      "color": "#ffffff"
    }
  },
  "edges": {
    "color": {
      "color": "#ffffff",
      "highlight": "#91cefc"
    },
    "smooth": {
      "type": "continuous"
    }
  },
  "physics": {
    "enabled": false
  },
  "interaction": {
    "hover": true,
    "tooltipDelay": 100,
    "hoverConnectedEdges": true,
    "multiselect": true,
    "selectable": false
  }
}
""")

# ----------------------------
# 9. Export HTML
# ----------------------------
html_file = "anime_community_network.html"
net.write_html(html_file, notebook=False)

# ----------------------------
# 10. Inject JS (Physics ON after load, title only)
# ----------------------------
custom_js = """
<script type="text/javascript">
(function() {
    // Physics ON after load (simplified for speed)
    window.addEventListener("load", function() {
        network.setOptions({
            physics: {
                enabled: true,
                barnesHut: {
                    gravitationalConstant: -2000,
                    centralGravity: 0.1,
                    springLength: 200,
                    springConstant: 0.005,
                    damping: 0.4,
                    avoidOverlap: 0.5
                },
                stabilization: false
            }
        });
    });

    // Network title
    var titleDiv = document.createElement('div');
    titleDiv.innerHTML = "Anime-Community Bipartite Network";
    titleDiv.style.position = 'absolute';
    titleDiv.style.top = '20px';
    titleDiv.style.left = '50%';
    titleDiv.style.transform = 'translateX(-50%)';
    titleDiv.style.color = '#61df90';
    titleDiv.style.fontSize = '18px';
    titleDiv.style.fontWeight = 'bold';
    titleDiv.style.fontFamily = 'Tahoma';
    document.body.appendChild(titleDiv);

})();
</script>
"""

with open(html_file, "r", encoding="utf-8") as f:
    html_content = f.read()

html_content = html_content.replace("</body>", custom_js + "\n</body>")

with open(html_file, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"Interactive bipartite network saved to {html_file} (gradient bridge anime, green Action & Adventure, tooltips show communities)")

Interactive bipartite network saved to anime_community_network.html (gradient bridge anime, green Action & Adventure, tooltips show communities)


Making Network3 html:

In [46]:
import pandas as pd

df = pd.read_csv("about_networks/user_animelist_sampled.csv")
print(df.columns)

Index(['user_id', 'anime_id', 'rating'], dtype='str')


In [60]:
from pyvis.network import Network
import pandas as pd
import networkx as nx

# ----------------------------
# 1. Load Data
# ----------------------------
df = pd.read_csv("about_networks/user_animelist_sampled.csv")[['user_id','anime_id']]
anime_comm_df = pd.read_csv("about_networks/anime_community_assignments.csv")[
    ['anime_id','all_community_names']
]

# Explode anime → community
anime_comm_df['all_community_names'] = anime_comm_df['all_community_names'].str.split(',')
anime_comm_exploded = anime_comm_df.explode('all_community_names')
anime_comm_exploded['all_community_names'] = anime_comm_exploded['all_community_names'].str.strip()

# Merge user → anime with anime → community
user_comm = df.merge(anime_comm_exploded, on='anime_id', how='left')

# ----------------------------
# 2. Compute top community per user
# ----------------------------
user_counts = (
    user_comm.groupby(['user_id','all_community_names'])
    .size()
    .reset_index(name='count')
)

user_totals = (
    user_counts.groupby('user_id')['count']
    .sum()
    .reset_index(name='total')
)

user_counts = user_counts.merge(user_totals, on='user_id')
user_counts['perc'] = user_counts['count'] / user_counts['total']

# Keep only top community per user
top_users = user_counts.loc[user_counts.groupby('user_id')['perc'].idxmax()]

# ----------------------------
# 3. Create Graph
# ----------------------------
G = nx.Graph()

# Distinct community colors
community_colors = {
    "Action & Adventure": "#9145b0",
    "Romance & Slice of Life": "#0fceaa",
    "Dark & Psychological": "#f37cd6",
    "Niche & Fantasy": "#0e149e"
}

comm_sizes = anime_comm_exploded['all_community_names'].value_counts().to_dict()

for comm, color in community_colors.items():
    G.add_node(
        comm,
        node_type='community',
        color=color,
        size=85,  # large enough for text
        physics=False,
        label=comm,
        shape="dot",
        font={
            "color": "white",
            "size": 18,
            "face": "Tahoma",
            "bold": True
        },
        title=f"<b>Community:</b> {comm}<br><b>#Anime:</b> {comm_sizes.get(comm,0)}"
    )

# ----------------------------
# 4. Add Sampled Users
# ----------------------------
SAMPLE_FRACTION = 0.5
EDGE_THRESHOLD = 0.10

top_users_sampled = top_users.sample(frac=SAMPLE_FRACTION, random_state=42)

for row in top_users_sampled.itertuples(index=False):
    user, comm, count, total, perc = row
    
    if perc < EDGE_THRESHOLD:
        continue

    G.add_node(
        user,
        node_type='user',
        color="#9b00ff",  # bright purple
        size=12,
        physics=True,
        title=f"<b>User:</b> {user}<br><b>% Completed:</b> {perc:.1%}"
    )

    alpha = 0.3 + 0.7 * perc
    width = 1 + 4 * perc
    edge_color = f"rgba(255,255,255,{alpha:.2f})"

    G.add_edge(user, comm, value=perc, color=edge_color, width=width)

# ----------------------------
# 5. Create PyVis Network
# ----------------------------
net = Network(
    height="800px",
    width="100%",
    bgcolor="#111111",
    font_color="white"
)

net.from_nx(G)

# Fixed positions for communities
positions = {
    "Action & Adventure": (-8000, 5000),
    "Romance & Slice of Life": (8000, -5000),
    "Dark & Psychological": (-3000, -3000),
    "Niche & Fantasy": (8000, 5000)
}

for node in net.nodes:
    if node["id"] in positions:
        node["x"], node["y"] = positions[node["id"]]
        node["physics"] = False

# Visual settings
net.set_options("""
{
  "nodes": {
    "borderWidth": 2,
    "font": {
      "color": "#ffffff",
      "size": 14,
      "face": "Tahoma",
      "strokeWidth": 0
    }
  },
  "edges": {
    "smooth": {"type": "continuous"}
  },
  "physics": {
    "enabled": false
  },
  "interaction": {
    "hover": true,
    "tooltipDelay": 100,
    "hoverConnectedEdges": true,
    "multiselect": false,
    "selectable": false
  }
}
""")

html_file = "user_community_network.html"
net.write_html(html_file, notebook=False)

# ----------------------------
# 6. Light Physics After Load
# ----------------------------
custom_js = """
<script type="text/javascript">
(function(){
    window.addEventListener("load", function() {
        network.setOptions({
            physics:{
                enabled:true,
                barnesHut:{
                    gravitationalConstant:-1500,
                    centralGravity:0.1,
                    springLength:200,
                    springConstant:0.005,
                    damping:0.4,
                    avoidOverlap:0.5
                },
                stabilization:false
            }
        });
    });

    var titleDiv = document.createElement('div');
    titleDiv.innerHTML="User-Top Community Bipartite Network";
    titleDiv.style.position='absolute';
    titleDiv.style.top='20px';
    titleDiv.style.left='50%';
    titleDiv.style.transform='translateX(-50%)';
    titleDiv.style.color='#9b00ff';
    titleDiv.style.fontSize='18px';
    titleDiv.style.fontWeight='bold';
    titleDiv.style.fontFamily='Tahoma';
    document.body.appendChild(titleDiv);
})();
</script>
"""

with open(html_file, "r", encoding="utf-8") as f:
    html_content = f.read()

html_content = html_content.replace("</body>", custom_js + "\n</body>")

with open(html_file, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"Final network saved to {html_file}")

Final network saved to user_community_network.html
